In [ ]:
from winml import register_execution_providers
register_execution_providers()

In [ ]:
from urllib import request

test_image_url = "https://github.com/facebookresearch/segment-anything/blob/main/notebooks/images/truck.jpg?raw=true"
test_image_name = "truck.jpg"

request.urlretrieve(test_image_url, test_image_name)

from IPython.display import Image, display

display(Image(filename=test_image_name))

In [ ]:
import ipywidgets as widgets
from IPython.display import display

dropdown = widgets.Dropdown(
    options=["OpenVINOExecutionProvider", "QNNExecutionProvider"],
    value="OpenVINOExecutionProvider",
    description="Execution Provider:",
)
display(dropdown)
ExecutionProvider="OpenVINOExecutionProvider"
ExecutionProvider = dropdown.value

def on_ep_change(change):
    global ExecutionProvider
    ExecutionProvider = change['new']

dropdown.observe(on_ep_change, names='value')

In [ ]:
from IPython import get_ipython
from IPython.core.magic import register_cell_magic

@register_cell_magic
def skip_if(line, cell):
    """
    Skips execution if 'line' evaluates to True.

    Args:
        line (str): boolean expression to apply eval(line) on
        cell (str): code to execute if eval(line) is False
    """
    if eval(line):
        return
    get_ipython().run_cell(cell)

In [ ]:
%%skip_if ExecutionProvider != "QNNExecutionProvider"
encoder_path = "./model/encoder/model.onnx"
decoder_path = "./model/decoder/model.onnx"

In [ ]:
%%skip_if ExecutionProvider != "OpenVINOExecutionProvider"
# define Olive generated OVIR ONNX Encapsulated model paths
from pathlib import Path
encoder_path = Path("models/encoder/sam21_encoder_pt_ov_int8_quant_st.onnx")
decoder_path = Path("models/decoder/sam21_decoder_pt_ov_int8_quant_st.onnx")

ov_device_dropdown = widgets.Dropdown(
    options=["CPU", "GPU", "NPU"],
    value="NPU",
    description=f"{ExecutionProvider} Device:",
)
display(ov_device_dropdown)
ov_device = ov_device_dropdown.value

def on_device_change(change):
    global ov_device
    ov_device = change['new']

ov_device_dropdown.observe(on_device_change, names='value')

In [ ]:
%%skip_if ExecutionProvider != "QNNExecutionProvider"
import sys
from pathlib import Path
import onnxruntime as ort

NOTEBOOK_DIR = Path(__file__).parent if "__file__" in globals() else Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parents[1]
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
from PIL import Image
from sam2_mask_generator import get_mask_ort

def add_ep_for_device(session_options, ep_name, device_type, ep_options=None):
    ep_devices = ort.get_ep_devices()
    for ep_device in ep_devices:
        if ep_device.ep_name == ep_name and ep_device.device.type == device_type:
            print(f"Adding {ep_name} for {device_type}")
            session_options.add_provider_for_devices([ep_device], {} if ep_options is None else ep_options)
            break


sess_options = ort.SessionOptions()

add_ep_for_device(sess_options, ExecutionProvider, ort.OrtHardwareDeviceType.CPU)

# Load image
raw_image = Image.open(test_image_name).convert("RGB")
input_box = [[[100, 300], [1750, 900]]]

# Load models
sess_ve = ort.InferenceSession(encoder_path, sess_options=sess_options)
sess_md = ort.InferenceSession(decoder_path, sess_options=sess_options)

sess_ve_inputs = sess_ve.get_inputs()
sess_md_inputs = sess_md.get_inputs()

ve_dtype = np.float32 if sess_ve_inputs[0].type == 'tensor(float)' else np.float16
md_dtype = np.float32 if sess_md_inputs[0].type == 'tensor(float)' else np.float16

# Get mask
mask = get_mask_ort(sess_ve, sess_md, raw_image, input_box, ve_dtype, md_dtype, sess_ve_inputs, sess_md_inputs)

# Save mask using PIL
mask_img = Image.fromarray(mask * 255)  # Convert binary mask to 0-255

from IPython.display import display
display(mask_img)

In [ ]:
%%skip_if ExecutionProvider != "OpenVINOExecutionProvider"
# open the image in PIL
from PIL import Image as PILImage
from ov_inference_utils import FacebookSam2_1HieraSmall, IPromptedImageSegmentationModel

input_image = PILImage.open(test_image_name)

# define the foreground, background and bbox for segmentation
foreground_points = [(500, 375), (1125, 625), (575, 750), (1405, 575)]
background_points = []
bbox = None

# create and instantiate the ORT model sessions for the vision encoder and mask decoder
model = FacebookSam2_1HieraSmall(
    model_path = {
        "vision_encoder": encoder_path,
        "mask_decoder": decoder_path
    },
    device=ov_device,
    execution_provider=ExecutionProvider
)

# run forward pass with the input image and prompts to generate the prediction using the segment_image function of the model class
prediction = model.segment_image(
    image=input_image,
    prompt=IPromptedImageSegmentationModel.Prompt(
        bbox=bbox,
        foreground_points=foreground_points,
        background_points=background_points
    )
)

# Plot the image + bbox + foreground and background points + predicted mask
import numpy as np
from IPython.display import display

# convert input PIL image to numpy array
image_array = np.array(input_image)

if prediction.mask is not None:
    print(f"prediction.mask shape: {prediction.mask.shape}")
    mask_array = prediction.mask

    # Alpha-blend mask overlay onto image
    overlay = image_array.copy().astype(np.float32)
    mask_rgb = np.zeros_like(image_array, dtype=np.float32)
    mask_rgb[mask_array > 0] = [0, 100, 255]
    alpha = 0.75
    overlay[mask_array > 0] = (
        overlay[mask_array > 0] * (1 - alpha) + mask_rgb[mask_array > 0] * alpha
    )
    overlay = overlay.astype(np.uint8)

    # Draw bbox on overlay
    if bbox:
        x, y, w, h = bbox
        thickness = 2
        overlay[y:y+h, x:x+thickness] = [0, 0, 255]
        overlay[y:y+h, x+w-thickness:x+w] = [0, 0, 255]
        overlay[y:y+thickness, x:x+w] = [0, 0, 255]
        overlay[y+h-thickness:y+h, x:x+w] = [0, 0, 255]

    # Draw foreground points on overlay
    for pt in foreground_points:
        x, y = int(pt[0]), int(pt[1])
        rr, cc = np.ogrid[max(0, y-6):min(overlay.shape[0], y+7), max(0, x-6):min(overlay.shape[1], x+7)]
        mask_circle = (rr - y)**2 + (cc - x)**2 <= 36
        overlay[max(0, y-6):min(overlay.shape[0], y+7), max(0, x-6):min(overlay.shape[1], x+7)][mask_circle] = [0, 255, 0]

    # Draw background points on overlay
    for pt in background_points:
        x, y = int(pt[0]), int(pt[1])
        rr, cc = np.ogrid[max(0, y-6):min(overlay.shape[0], y+7), max(0, x-6):min(overlay.shape[1], x+7)]
        mask_circle = (rr - y)**2 + (cc - x)**2 <= 36
        overlay[max(0, y-6):min(overlay.shape[0], y+7), max(0, x-6):min(overlay.shape[1], x+7)][mask_circle] = [255, 0, 0]

    display(PILImage.fromarray(overlay))
else:
    print("No mask predicted.")
    display(PILImage.fromarray(image_array))